In [ ]:
""" --- Integration test ---
 Author: Fuad Godzhaev
 Integration of multiple functions we have so far and testing if they work together
 The step-by-step process is as follows:
 1) Pre-processing: denoising, rotation, contrast adjustments
 2) Text-detection: segmentation of areas containing text
 3) Character recognition: breaking down highlited text region into individual characters for recognition

 TODO:
 -Check for colored/grey images; implement different behaviour
 -More robust edges and contour detection
 -Check for the image needing rotation (i.e. img3)
"""

import cv2
import numpy as np
import easygui as eg
from matplotlib import pyplot as plt

# 2. Pre-processing
def pre_processing(img):

    l, a, b = cv2.split(cv2.cvtColor(img, cv2.COLOR_BGR2LAB))

    blurred = cv2.GaussianBlur(l, (51,51), 0)

    laplacian = cv2.Laplacian(blurred, cv2.CV_64F)
    laplacian = cv2.convertScaleAbs(laplacian)

    subtracted = cv2.subtract(l, laplacian)
    normalized = cv2.normalize(subtracted, None, 0, 255, cv2.NORM_MINMAX)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(5,5))
    l_enh = clahe.apply(normalized)

    # Applying denoising filter
    denoised = cv2.fastNlMeansDenoising(l_enh, None, 3, 7, 21)

    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
    opening = cv2.morphologyEx(denoised, cv2.MORPH_OPEN, kernel)
    opening = cv2.morphologyEx(opening, cv2.MORPH_CLOSE, kernel, 2)
    
    inverted = cv2.bitwise_not(opening)

    # Adaptive thresholding for local contrast handling, works better than global threshold for uneven lighting
    #thresh = cv2.threshold(inverted, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY)
    _, thresh = cv2.threshold(inverted, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    return(thresh)

def text_detection(img):
    # Apply global threshold (Otsu)
    _, binary = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Morphological cleanup (remove small noise and fill gaps)
    kernel = np.ones((3,3), np.uint8)
    #cleaned = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)
    cleaned = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, kernel)

    return(cleaned)

def character_recognition(img):
    pass



# 1. Load image (replace with your handwritten image)
#f = eg.fileopenbox(msg="Choose a file to open", title="Open Image", filetypes=["*.jpg","*.jpeg","*.png", "*.tiff"])
img = cv2.imread("C:\\Users\\Wirexia\\Documents\\GitHub\\Script2Text\\Images\\sample.jpg")

cv2.namedWindow("Image", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Image", 1000, 1000)
cv2.imshow("Image", pre_processing(img))

cv2.waitKey(0)
cv2.destroyAllWindows()